# Poisson Ridge Regression — Llama 3.1-8B Embeddings (Layer 18)

Runs the Poisson ridge encoding model using Llama 3.1-8B layer-18 embeddings (most aligned model/layer from the geometry paper).  
Outputs: **beta fits** (per-neuron coefficient matrices) and **self-other beta correlations** across all 10 patients.  

**Sections**
1. Imports & Config
2. Llama embedding loader (from `.npy` cache)
3. Spike loader + Y-matrix builder
4. Regression functions
5. Main pipeline loop (one patient at a time)
6. Aggregate beta correlations
7. Reliability analysis

## 1 — Imports & Config

In [ ]:
import os
import sys
import importlib.util

PROJECT_ROOT = "/scratch/aniluchavez/hippocampal-speaker-semantics"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


RELIABILITY_MODULE_PATH = os.path.join(PROJECT_ROOT, "neural_encoding", "reliability.py")

def load_reliability_module(force_reload=False):
    """Load reliability.py directly, bypassing neural_encoding/__init__.py."""
    module_name = "nn_reliability"
    if force_reload and module_name in sys.modules:
        del sys.modules[module_name]
    if module_name in sys.modules:
        return sys.modules[module_name]
    spec = importlib.util.spec_from_file_location(module_name, RELIABILITY_MODULE_PATH)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

import pickle
import warnings
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor
from sklearn.model_selection import KFold
from scipy.special import gammaln
import statsmodels.api as sm

from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.simplefilter('ignore', PerfectSeparationWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

try:
    from sklearnex import patch_sklearn
    patch_sklearn()
except ImportError:
    pass


In [ ]:
# ============================================================
# CONFIG — edit here only
# ============================================================

# Embedding source
EMBED_CACHE_DIR  = "/scratch/aniluchavez/ConvoDATAS/EmbedCache"
BERT_EMBED_DIR   = "/scratch/aniluchavez/ConvoDATAS/BERTEmbeds"   # used only for word/speaker metadata
MODEL_TAG        = "llama-3.1-8b"
LLAMA_LAYER      = 18   # peak layer (hippocampus, geometry paper)

# Pipeline parameters
N_COMPONENTS         = 30
TARGET_SPEAKER       = "SPK1"
N_ITERATIONS         = 10
N_X_SHUFFLE_NULLS     = 20   # was hard-coded at 100; use 0-20 for smoke tests, 100+ for final nulls
N_JOBS               = -1
N_JOBS_RELIABILITY   = 10
N_NULLS_RELIABILITY  = 100
ALPHAS               = np.logspace(-3, 2, 10)
SPEAKERS             = [f"Speaker{i}" for i in range(1, 13)]

# Output directory
RESULTS_ROOT = "/projects/bhayden/anilu/Language_docs/RegressionRESULTSLLAMA31_L18/allwords"
if RESULTS_ROOT.startswith("/projects") and not os.path.isdir("/projects"):
    RESULTS_ROOT = "/scratch/aniluchavez/RegressionRESULTSLLAMA31_L18/allwords"
os.makedirs(RESULTS_ROOT, exist_ok=True)
print(f"RESULTS_ROOT: {RESULTS_ROOT}")


# Spike-window extraction
SPIKE_SOURCE_MODE = "extract"  # "extract" regenerates from .mat when needed; "cached" only loads existing .npy files
REGENERATE_SPIKE_WINDOWS = False
RESUME_COMPLETED_PATIENTS = True
SPIKE_MAT_ROOT = "/scratch/aniluchavez/ConvoDATAS/SpikesMAT"
SPIKE_WINDOW_OUTPUT_ROOT = "/scratch/aniluchavez/ConvoDATAS/SpikeWindows"
SPIKE_SAMPLE_RATE = 1000  # Hz; onset/offset files are in ms
SPIKE_VALUE_MODE = "counts"  # "counts" for Poisson Y, or "rates" for spikes/s
SPIKE_WINDOW_PRESETS = {
    "self":  {"speaker": "Speaker1", "start_offset_ms": -150, "window_length_ms": 500},
    "other": {"start_offset_ms":  200, "window_length_ms": 500},
}
REGIONS = ["hippocampus", "ACC"]


# Patient list — region_ranges map channel numbers → brain regions (patient-specific electrode placements)
# Regions present in each patient's region_ranges are the ones that will be run.
TRANSCRIPTS_DIR = "/scratch/aniluchavez/ConvoDATAS/Transcripts"

PATIENTS = [
    {"patient_ID": "PTYEU_task147", "patient": "ptYEU_task147",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(17,24),(41,48)]}},
    {"patient_ID": "PTYFF_task17",  "patient": "ptYFF_task17",
     "region_ranges": {"hippocampus": [(9,16),(25,40)], "ACC": [(17,24),(41,48)]}},
    {"patient_ID": "PTYFG_task18",  "patient": "ptYFG_task18",
     "region_ranges": {"hippocampus": [(9,16)],         "ACC": [(25,56)]}},
    {"patient_ID": "PTYFI_task81",  "patient": "ptYFI_task81",
     "region_ranges": {"hippocampus": [(1,8),(25,40)],  "ACC": [(9,16)]}},
    {"patient_ID": "PTYFA_task25",  "patient": "ptYFA_task25",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(17,24)]}},
    {"patient_ID": "PTYFK_task40",  "patient": "ptYFK_task40",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(49,56)]}},
    {"patient_ID": "PTYEY_task86",  "patient": "ptYEY_task86",
     "region_ranges": {"hippocampus": [(1,16)]}},
    {"patient_ID": "PTYEV_task37",  "patient": "ptYEV_task37",
     "region_ranges": {"hippocampus": [(1,16),(25,40)], "ACC": [(17,24),(41,48)]}},
    {"patient_ID": "PTYEZ_task60",  "patient": "ptYEZ_task60",
     "region_ranges": {"hippocampus": [(1,16)],         "ACC": [(17,24)]}},
    {"patient_ID": "PTYFC_task28",  "patient": "ptYFC_task28",
     "region_ranges": {"hippocampus": [(1,8),(33,48)],  "ACC": [(17,32),(49,64)]}},
    {"patient_ID": "PTYFM_task104", "patient": "ptYFM_task104",
     "region_ranges": {"EC":          [(1,8),(25,32)],          "thalamus":    [(9,16)],
                       "OFC":         [(17,24),(49,56)],         "hippocampus": [(33,48)]}},
    {"patient_ID": "PTYFP_task88",  "patient": "ptYFP_task88",
     "region_ranges": {"OFC":         [(1,8),(33,40)],           "ACC":         [(9,16),(41,48)],
                       "hippocampus": [(17,24),(25,32),(49,56),(57,64)]}},
    {"patient_ID": "PTYFR_task91",  "patient": "ptYFR_task91",
     "region_ranges": {"hippocampus": [(1,16),(41,56)],          "AMY":         [(17,24),(33,40)],
                       "ACC":         [(25,32),(57,64)]}},
    {"patient_ID": "PTYFS_task95",  "patient": "ptYFS_task95",
     "region_ranges": {"hippocampus": [(1,24)]}},
    {"patient_ID": "PTYFU_task224", "patient": "ptYFU_task224",
     "region_ranges": {"AMY":         [(1,8),(33,40)],           "ACC":         [(9,16),(57,64)],
                       "hippocampus": [(17,32),(41,56)]}},
]


## 2 — Llama Embedding Loader

In [ ]:
def load_and_reduce_llama_embeddings(patient_ID, n_components=N_COMPONENTS):
    """
    Load Llama layer embeddings from .npy cache and reduce with PCA.
    Uses BERT CSV only for word/speaker metadata (same word order).
    Returns: df_metadata (Word, Speaker, RowIndex), pcs_df (n_words x n_components)
    """
    bert_csv = os.path.join(
        BERT_EMBED_DIR, f"{patient_ID}_words_english_only",
        f"{patient_ID}_aligned_embeddings_withNP.csv"
    )
    df_metadata = pd.read_csv(bert_csv)[["Word", "Speaker", "RowIndex"]].reset_index(drop=True)

    npy_path = os.path.join(EMBED_CACHE_DIR, f"{patient_ID}_{MODEL_TAG}_word_emb_layers.npy")
    arr = np.load(npy_path)                    # (n_layers, n_words, embed_dim)
    emb = arr[LLAMA_LAYER].astype(np.float32)  # (n_words, 4096)

    if emb.shape[0] != len(df_metadata):
        raise ValueError(
            f"{patient_ID}: embedding rows ({emb.shape[0]}) != metadata rows ({len(df_metadata)})"
        )

    pca = PCA(n_components=n_components)
    pcs = pca.fit_transform(emb)
    pcs_df = pd.DataFrame(pcs, columns=[f"PC{i+1}" for i in range(n_components)])

    print(f"  {patient_ID}: Llama layer {LLAMA_LAYER}, emb {emb.shape} → PCA {pcs.shape}")
    return df_metadata, pcs_df


def build_feature_matrix_core(pcs_df, durations):
    """PCA features only — duration excluded (fixed 500ms window, constant across all words)."""
    X = pcs_df.values
    if X.shape[0] == 0:
        raise ValueError("No samples for selected speaker subset.")
    return StandardScaler().fit_transform(X)


def get_self_and_other_features(patient_ID, duration_file, n_components=N_COMPONENTS, target_speaker=TARGET_SPEAKER):
    """Return X_self, X_other, df_metadata, words_self, words_other."""
    df_metadata, pcs_df = load_and_reduce_llama_embeddings(patient_ID, n_components)

    if duration_file.endswith(".xlsx"):
        dur_df = pd.read_excel(duration_file)
    else:
        dur_df = pd.read_csv(duration_file)
    dur_df = dur_df.reset_index(drop=True)
    df_metadata["regress_dur"] = dur_df["regress_dur"]
    print(f"  Duration file rows: {len(dur_df)}")

    mask_self  = df_metadata["Speaker"] == target_speaker
    mask_other = df_metadata["Speaker"] != target_speaker

    X_self  = build_feature_matrix_core(pcs_df[mask_self].reset_index(drop=True),
                                        df_metadata.loc[mask_self,  "regress_dur"].reset_index(drop=True))
    X_other = build_feature_matrix_core(pcs_df[mask_other].reset_index(drop=True),
                                        df_metadata.loc[mask_other, "regress_dur"].reset_index(drop=True))

    words_self  = df_metadata.loc[mask_self,  "Word"].reset_index(drop=True)
    words_other = df_metadata.loc[mask_other, "Word"].reset_index(drop=True)

    print(f"  X_self: {X_self.shape}  X_other: {X_other.shape}")
    return X_self, X_other, df_metadata, words_self, words_other

## 3 — Spike Window Extraction from `.mat`

Use this section when the fixed-window spike matrices have not already been cached. The default window is 150 ms before Speaker1 word onset for 500 ms, and 200 ms after all other speaker word onsets for 500 ms.


In [ ]:
def get_patient_mat_path(patient):
    """Return the raw v7.3 MATLAB spike file for a patient label like ptYEU_task147."""
    patient_code = patient[2:5].upper()
    return os.path.join(SPIKE_MAT_ROOT, patient_code, f"{patient}_new_spikes.mat")


def get_timing_file(patient_ID):
    """Return word-timing xlsx: checks BERTEmbeds first (old patients), then Transcripts (new patients)."""
    timing_dir = os.path.join(BERT_EMBED_DIR, f"{patient_ID}_words_english_only")
    if os.path.isdir(timing_dir):
        preferred = os.path.join(timing_dir, f"{patient_ID}_filtered_used_rows_withNP.xlsx")
        if os.path.exists(preferred):
            return preferred
        candidates = sorted(
            os.path.join(timing_dir, f) for f in os.listdir(timing_dir)
            if f.endswith(".xlsx")
            and "filtered_used_rows_withNP" in f
            and not f.startswith(("._", "~$"))
        )
        if candidates:
            return candidates[0]
    transcripts_path = os.path.join(TRANSCRIPTS_DIR, f"{patient_ID}_filtered_used_rows_withNP_withClusterIDNew.xlsx")
    if os.path.exists(transcripts_path):
        return transcripts_path
    raise FileNotFoundError(f"No timing Excel found for {patient_ID}")


def get_spike_window_output_dir(patient):
    self_cfg = SPIKE_WINDOW_PRESETS["self"]
    other_cfg = SPIKE_WINDOW_PRESETS["other"]
    tag = (
        f"tshift{self_cfg['start_offset_ms']:+d}_tlen{self_cfg['window_length_ms']}"
        f"_oshift{other_cfg['start_offset_ms']:+d}_olen{other_cfg['window_length_ms']}"
    )
    return os.path.join(SPIKE_WINDOW_OUTPUT_ROOT, f"output_{patient}_english_only_{tag}")


def get_spike_duration_file(patient):
    return os.path.join(get_spike_window_output_dir(patient), f"{patient}_with_regress_dur.xlsx")


def speaker_window_config_for(speaker):
    if speaker == SPIKE_WINDOW_PRESETS["self"].get("speaker", "Speaker1"):
        return SPIKE_WINDOW_PRESETS["self"]
    return SPIKE_WINDOW_PRESETS["other"]


def load_mat_spikes_sparse(mat_path):
    """Load MATLAB v7.3 sparse spike matrix as CSR with shape time x neurons."""
    import h5py
    import scipy.sparse

    with h5py.File(mat_path, "r") as mat_file:
        spike_group = mat_file["spikes"]
        data = spike_group["data"][:]
        ir = spike_group["ir"][:]
        jc = spike_group["jc"][:]
        spikes = scipy.sparse.csr_matrix((data, ir, jc))
        qual = np.asarray(mat_file["qual"][()] if "qual" in mat_file else mat_file["quality"][()]).ravel()
        chan = np.asarray(mat_file["chan"][()] if "chan" in mat_file else mat_file["channelIds"][()]).ravel()

    print(f"  Loaded raw spikes: {spikes.shape} (time x neurons)")
    return spikes, qual, chan


def get_cells_by_region(chan, qual, region_ranges, good_qual=(4, 5)):
    region_cells = {}
    for region, ranges in region_ranges.items():
        selected = []
        for min_chan, max_chan in ranges:
            in_range = np.where((chan >= min_chan) & (chan <= max_chan))[0]
            good = in_range[np.isin(qual[in_range], good_qual)]
            selected.extend(good.tolist())
        region_cells[region] = np.array(selected, dtype=int)
        print(f"  {region}: {len(region_cells[region])} good cells")
    return region_cells


def build_spike_window_table(timing_file, output_duration_file=None):
    """Add fixed-window start/end columns and regress_dur for the model."""
    xl = pd.ExcelFile(timing_file)
    sheet = "Sheet1" if "Sheet1" in xl.sheet_names else xl.sheet_names[0]
    df = xl.parse(sheet)
    df.columns = [c.strip().replace("’", "").replace("‘", "").replace("“", "").replace("”", "") for c in df.columns]
    speaker_cols = [c for c in df.columns if c.lower().startswith("speaker")]
    if not speaker_cols:
        raise ValueError(f"No Speaker* columns found in {timing_file}")

    if "CleanedWord" not in df.columns:
        df["CleanedWord"] = df[speaker_cols].apply(
            lambda row: next((str(v).strip() for v in row if str(v).strip() not in ("", "nan")), ""),
            axis=1
        )

    starts, ends, durations, speakers = [], [], [], []
    for _, row in df.iterrows():
        speaker = next((spk for spk in speaker_cols if pd.notna(row.get(spk)) and str(row.get(spk)).strip() not in ("", "nan")), None)
        onset = row.get("onset")
        if speaker is None or pd.isna(onset):
            starts.append(np.nan)
            ends.append(np.nan)
            durations.append(np.nan)
            speakers.append(np.nan)
            continue

        cfg = speaker_window_config_for(speaker)
        start = float(onset) + float(cfg["start_offset_ms"])
        end = start + float(cfg["window_length_ms"])
        starts.append(start)
        ends.append(end)
        durations.append(float(cfg["window_length_ms"]))
        speakers.append(speaker)

    df["window_speaker"] = speakers
    df["word_dur"] = df["offset"] - df["onset"] if {"offset", "onset"}.issubset(df.columns) else np.nan
    df["regress_dur"] = durations
    df["regress_pre_onset"] = starts
    df["regress_post_offset"] = ends

    if output_duration_file is not None:
        os.makedirs(os.path.dirname(output_duration_file), exist_ok=True)
        df.to_excel(output_duration_file, index=False)
        print(f"  Saved duration/window table: {output_duration_file}")
    return df


def extract_spike_windows_to_cache(spikes, region_cells, window_df, output_dir):
    """Save per-speaker, per-region fixed-window spike matrices for load_spike_data()."""
    os.makedirs(output_dir, exist_ok=True)
    speaker_cols = [c for c in window_df.columns if c.lower().startswith("speaker")]
    n_time = spikes.shape[0]

    for speaker in speaker_cols:
        rows = window_df[window_df[speaker].notna()].copy()
        if rows.empty:
            continue

        speaker_dir = os.path.join(output_dir, speaker)
        os.makedirs(speaker_dir, exist_ok=True)
        starts = rows["regress_pre_onset"].to_numpy(dtype=float)
        ends = rows["regress_post_offset"].to_numpy(dtype=float)
        lengths = rows["regress_dur"].to_numpy(dtype=float)

        for region, neuron_indices in region_cells.items():
            if len(neuron_indices) == 0:
                continue

            mat = np.full((len(rows), len(neuron_indices)), np.nan, dtype=float)
            for i, (start_ms, end_ms, dur_ms) in enumerate(zip(starts, ends, lengths)):
                if np.isnan(start_ms) or np.isnan(end_ms) or dur_ms <= 0:
                    continue
                start_idx = int(round(start_ms * SPIKE_SAMPLE_RATE / 1000.0))
                end_idx = int(round(end_ms * SPIKE_SAMPLE_RATE / 1000.0))
                if start_idx < 0 or end_idx > n_time or end_idx <= start_idx:
                    continue

                counts = np.asarray(spikes[start_idx:end_idx, neuron_indices].sum(axis=0)).ravel()
                if SPIKE_VALUE_MODE == "rates":
                    mat[i, :] = counts * (1000.0 / dur_ms)
                elif SPIKE_VALUE_MODE == "counts":
                    mat[i, :] = counts
                else:
                    raise ValueError(f"Unsupported SPIKE_VALUE_MODE: {SPIKE_VALUE_MODE}")

            save_path = os.path.join(speaker_dir, f"{region}_spike_{SPIKE_VALUE_MODE}.npy")
            np.save(save_path, mat)
            print(f"  Saved {speaker}-{region}: {mat.shape} -> {save_path}")


def prepare_spike_windows_for_patient(patient_ID, patient, region_ranges):
    """Create or reuse fixed-window spike matrices and the matching duration table."""
    output_dir = get_spike_window_output_dir(patient)
    duration_file = get_spike_duration_file(patient)
    expected_file = os.path.join(output_dir, "Speaker1")

    if (
        SPIKE_SOURCE_MODE == "cached"
        or (not REGENERATE_SPIKE_WINDOWS and os.path.isdir(expected_file) and os.path.exists(duration_file))
    ):
        print(f"  Using cached spike windows: {output_dir}")
        return output_dir, duration_file

    timing_file = get_timing_file(patient_ID)
    mat_path = get_patient_mat_path(patient)
    if not os.path.exists(mat_path):
        raise FileNotFoundError(f"Raw spike .mat not found: {mat_path}")

    print(f"  Building spike windows from: {mat_path}")
    print(f"  Timing source: {timing_file}")
    window_df = build_spike_window_table(timing_file, output_duration_file=duration_file)
    spikes, qual, chan = load_mat_spikes_sparse(mat_path)
    region_cells = get_cells_by_region(chan, qual, region_ranges)
    extract_spike_windows_to_cache(spikes, region_cells, window_df, output_dir)
    return output_dir, duration_file


## 4 — Spike Loader + Y-Matrix Builder


In [ ]:
def load_spike_data(speakers, regions, self_base_dir, other_base_dir=None,
                    self_shift_tag=None, other_shift_tag=None,
                    target_speaker="Speaker1", mode="auto"):
    spike_data = {}
    for speaker in speakers:
        if mode == "split":
            base_dir  = self_base_dir  if speaker == target_speaker else other_base_dir
            shift_tag = self_shift_tag if speaker == target_speaker else other_shift_tag
        elif mode == "unified":
            base_dir  = self_base_dir
            shift_tag = self_shift_tag if speaker == target_speaker else other_shift_tag
        elif mode == "auto":
            base_dir  = self_base_dir
            shift_tag = None
        else:
            raise ValueError(f"Invalid mode '{mode}'.")

        try:
            all_folders = os.listdir(base_dir)
        except FileNotFoundError:
            print(f"[ERROR] Base dir not found: {base_dir}")
            continue

        folder = next((f for f in all_folders if f.strip().lower() == speaker.strip().lower()), None)
        if not folder:
            continue

        speaker_path = os.path.join(base_dir, folder)
        spike_data[speaker] = {}

        for region in regions:
            matched = False
            for variant in [region, region.upper(), region.lower()]:
                if mode == "auto":
                    candidates = [
                        f for f in os.listdir(speaker_path)
                        if f.lower().startswith(variant.lower()) and f.endswith(".npy")
                    ]
                    if candidates:
                        spike_data[speaker][region] = np.load(os.path.join(speaker_path, candidates[0]))
                        print(f"  Loaded {speaker}-{region}: {spike_data[speaker][region].shape}")
                        matched = True
                        break
                else:
                    fpath = os.path.join(speaker_path, f"{variant}_spike_counts_shift{shift_tag}.npy")
                    if os.path.exists(fpath):
                        spike_data[speaker][region] = np.load(fpath)
                        print(f"  Loaded {speaker}-{region}: {spike_data[speaker][region].shape}")
                        matched = True
                        break
                if matched:
                    break
            if not matched:
                pass  # speaker may not have this region; skip silently
    return spike_data


def build_Y_matrices(df_metadata, spike_data, regions, target_speaker=TARGET_SPEAKER):
    speaker_tag_map = {
        f"SPK{spk.replace('Speaker', '').strip()}": spk.strip()
        for spk in spike_data.keys()
    }
    region_Ys = {}

    for region in regions:
        Y_self, Y_other = [], []
        counters = {spk: 0 for spk in spike_data}

        for _, row in df_metadata.iterrows():
            spk_tag = row["Speaker"]
            spk_name = speaker_tag_map.get(spk_tag)
            if spk_name is None or region not in spike_data.get(spk_name, {}):
                continue

            idx = counters[spk_name]
            if idx >= spike_data[spk_name][region].shape[0]:
                continue
            row_data = spike_data[spk_name][region][idx, :]

            if spk_tag == target_speaker:
                Y_self.append(row_data)
            else:
                Y_other.append(row_data)
            counters[spk_name] += 1

        print(f"  {region}: Y_self={len(Y_self)}  Y_other={len(Y_other)}")
        region_Ys[region] = {
            "self":  np.vstack(Y_self)  if Y_self  else np.empty((0,)),
            "other": np.vstack(Y_other) if Y_other else np.empty((0,)),
        }
    return region_Ys

## 4 — Regression Functions

In [ ]:
def run_poisson_ridge(
    X, Y, patient_id, neuron_idx, region_name, n_semantic_dims,
    n_iterations=N_ITERATIONS, test_size=0.3, n_shuf_x=100, alpha_grid=None
):
    def safe_int(x):
        return int(x[0]) if isinstance(x, (list, tuple, np.ndarray)) else int(x)

    def poisson_ll(y_true, mu):
        mu = np.clip(mu, 1e-10, None)
        return float(np.sum(y_true * np.log(mu) - mu - gammaln(y_true + 1)))

    def effective_dof(Xm, mu, alpha):
        mu = np.clip(mu, 1e-8, None)
        Xw = Xm * np.sqrt(mu)[:, None]
        _, s, _ = np.linalg.svd(Xw, full_matrices=False)
        s2 = s ** 2
        return float(np.sum(s2 / (s2 + alpha)))

    def choose_alpha_cv(X_train, y_train, alphas, seed):
        kf = KFold(n_splits=5, shuffle=True, random_state=seed)
        best_alpha, best_ll = alphas[0], -np.inf
        for alpha in alphas:
            scores = []
            for tr, va in kf.split(X_train):
                m = PoissonRegressor(alpha=alpha, max_iter=1000).fit(X_train[tr], y_train[tr])
                scores.append(poisson_ll(y_train[va], m.predict(X_train[va])))
            avg = np.mean(scores) if scores else -np.inf
            if avg > best_ll:
                best_ll, best_alpha = avg, alpha
        return best_alpha

    def make_split(n, base_ts, iter_idx):
        ts = base_ts
        for _ in range(4):
            sp = int((1 - ts) * n)
            offset = (iter_idx * int(n * ts)) % n
            idx_range = np.roll(np.arange(n), -offset)
            train_idx, test_idx = idx_range[:sp], idx_range[sp:]
            if len(test_idx) > (X.shape[1] + 1):
                return train_idx, test_idx, ts
            ts = min(0.6, ts + 0.1)
        return train_idx, test_idx, ts

    if alpha_grid is None:
        alpha_grid = ALPHAS

    neuron_idx = safe_int(neuron_idx)
    n_words, n_neurons = Y.shape
    if neuron_idx >= n_neurons:
        raise IndexError(f"neuron_idx {neuron_idx} >= {n_neurons}")

    y = Y[:, neuron_idx].astype(float)
    X = np.asarray(X, dtype=float)
    if np.std(y) == 0 or np.all(y == 0):
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    results_per_iter, coef_pval_rows, deviance_records = [], [], []
    coef_accumulator, support_flags = [], set()

    for iter_idx in range(n_iterations):
        train_idx, test_idx, used_ts = make_split(n_words, test_size, iter_idx)
        X_train, y_train = X[train_idx], y[train_idx]
        X_test,  y_test  = X[test_idx],  y[test_idx]

        best_alpha  = choose_alpha_cv(X_train, y_train, alpha_grid, seed=safe_int(iter_idx))
        ridge_model = PoissonRegressor(alpha=best_alpha, max_iter=1000).fit(X_train, y_train)
        pred_train  = ridge_model.predict(X_train)
        pred_test   = ridge_model.predict(X_test)

        ll_real = poisson_ll(y_test, pred_test)
        ll_null = poisson_ll(y_test, np.full_like(y_test, np.mean(y_test)))
        pseudo_r2 = 1 - (ll_real / ll_null)
        r_train, _ = pearsonr(y_train, np.clip(pred_train, 1e-10, None))
        r_test, corr_p = pearsonr(y_test, np.clip(pred_test, 1e-10, None))
        edf = effective_dof(X_train, np.clip(pred_train, 1e-10, None), best_alpha)

        coef_accumulator.append(ridge_model.coef_)

        deviance_vals = 2 * (y_test * np.log((y_test + 1e-10) / np.clip(pred_test, 1e-10, None))
                             - (y_test - pred_test))
        for i, word_idx in enumerate(test_idx):
            deviance_records.append({
                "neuron": neuron_idx, "word_index": safe_int(word_idx),
                "iteration": safe_int(iter_idx), "deviance": float(deviance_vals[i])
            })

        # X-shuffle null
        ll_xshuf_list, edf_shuf_list = [], []
        for _ in range(n_shuf_x):
            Xs = X_train.copy()
            for c in range(min(n_semantic_dims, Xs.shape[1])):
                np.random.shuffle(Xs[:, c])
            ms = PoissonRegressor(alpha=best_alpha, max_iter=1000).fit(Xs, y_train)
            ll_xshuf_list.append(poisson_ll(y_test, ms.predict(X_test)))
            edf_shuf_list.append(effective_dof(Xs, np.clip(ms.predict(Xs), 1e-10, None), best_alpha))

        ll_xshuf = float(np.mean(ll_xshuf_list))
        edf_shuf = float(np.mean(edf_shuf_list))

        # GLM p-values
        try:
            glm = sm.GLM(y_train, sm.add_constant(X_train, has_constant="add"),
                         family=sm.families.Poisson()).fit()
            pvals = glm.pvalues[1:]
            for p_idx, (coef, pval) in enumerate(zip(ridge_model.coef_, pvals)):
                coef_pval_rows.append({
                    "neuron": neuron_idx, "predictor_index": safe_int(p_idx),
                    "iteration": safe_int(iter_idx), "coefficient": float(coef), "p_value": float(pval)
                })
                if pval < 0.05:
                    support_flags.add(p_idx)
        except Exception:
            pass

        results_per_iter.append({
            "neuron": neuron_idx, "iteration": safe_int(iter_idx),
            "best_alpha": float(best_alpha), "test_size_used": float(used_ts),
            "ll_real": float(ll_real), "ll_null": float(ll_null), "ll_xshuf": float(ll_xshuf),
            "ll_diff_xshuf": float(ll_real - ll_xshuf),
            "pseudo_r2": float(pseudo_r2), "pseudo_r2_shuf": float(1 - (ll_xshuf / ll_null)),
            "corr_train": float(r_train), "corr_test": float(r_test), "corr_p": float(corr_p),
            "edf": float(edf), "edf_shuf": float(edf_shuf),
            "AIC": float(2 * edf - 2 * ll_real), "AIC_xshuf": float(2 * edf_shuf - 2 * ll_xshuf),
            "p_val_ll_diff_xshuf": float(np.mean(np.array(ll_xshuf_list) >= ll_real))
        })

    df = pd.DataFrame(results_per_iter)
    summary_df = df.groupby("neuron").median(numeric_only=True).reset_index()

    coef_df_all = pd.DataFrame(coef_pval_rows)
    sig_rows = []
    if not coef_df_all.empty:
        for p_idx, grp in coef_df_all.groupby("predictor_index"):
            if grp["p_value"].median() < 0.05:
                sig_rows.append({
                    "neuron": neuron_idx, "predictor_index": safe_int(p_idx),
                    "mean_coefficient": float(grp["coefficient"].mean()),
                    "median_p_value": float(grp["p_value"].median()),
                    "times_significant": int((grp["p_value"] < 0.05).sum()),
                    "ridge_support": int(p_idx in support_flags)
                })
    final_sig_df = pd.DataFrame(sig_rows)

    mean_coef = np.mean(coef_accumulator, axis=0) if coef_accumulator else np.array([])
    ridge_df = pd.DataFrame([{
        "neuron": neuron_idx, "predictor_index": safe_int(p_idx),
        "mean_ridge_coef": float(v), "ridge_support": int(p_idx in support_flags)
    } for p_idx, v in enumerate(mean_coef)])

    coef_df = (
        pd.DataFrame([mean_coef], index=[neuron_idx],
                     columns=[f"Predictor_{i}" for i in range(len(mean_coef))])
        if len(mean_coef) > 0 else pd.DataFrame()
    )

    return df, summary_df, final_sig_df, ridge_df, coef_df

In [ ]:
def run_all_conditions_for_patient(
    X_dict, Y_dict, patient_id, region_list, n_semantic_dims,
    n_iterations=N_ITERATIONS, n_jobs=N_JOBS, results_root=RESULTS_ROOT,
    n_shuf_x=N_X_SHUFFLE_NULLS
):
    results_dict = {}

    for region in region_list:
        for condition in ["self", "other"]:
            key = f"{region}_{condition}"
            X = X_dict.get(key)
            Y = Y_dict.get(key)
            if X is None or Y is None:
                print(f"  Skipping {key}: no data")
                continue

            print(f"  Running {patient_id} — {key} ({Y.shape[1]} neurons)")
            n_neurons = Y.shape[1]

            def run_single(nid):
                return run_poisson_ridge(
                    X=X, Y=Y, patient_id=patient_id, neuron_idx=nid,
                    region_name=key, n_semantic_dims=n_semantic_dims,
                    n_iterations=n_iterations, n_shuf_x=n_shuf_x
                )

            outputs = Parallel(n_jobs=n_jobs)(
                delayed(run_single)(nid) for nid in range(n_neurons)
            )

            all_df, all_summary, all_sig, all_ridge, all_coef = [], [], [], [], []
            for nid, out in enumerate(outputs):
                if not (isinstance(out, tuple) and len(out) == 5):
                    continue
                df, summary_df, sig_df, ridge_df, coef_df = out
                try:
                    coef_df.index = [nid]
                except Exception:
                    pass
                all_df.append(df)
                all_summary.append(summary_df)
                all_sig.append(sig_df)
                all_ridge.append(ridge_df)
                all_coef.append(coef_df)

            results_dict[key] = {
                "df":          pd.concat(all_df,      ignore_index=True) if all_df      else pd.DataFrame(),
                "summary":     pd.concat(all_summary, ignore_index=True) if all_summary else pd.DataFrame(),
                "significant": pd.concat(all_sig,     ignore_index=True) if all_sig     else pd.DataFrame(),
                "ridge":       pd.concat(all_ridge,   ignore_index=True) if all_ridge   else pd.DataFrame(),
                "coef":        pd.concat(all_coef)                        if all_coef    else pd.DataFrame(),
            }

    # Save
    patient_dir = os.path.join(results_root, patient_id)
    os.makedirs(patient_dir, exist_ok=True)
    save_path = os.path.join(patient_dir, "ALL_CONDITIONS_RESULTS.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(results_dict, f)
    print(f"  Saved: {save_path}")

    return results_dict


## 5 — Main Pipeline

In [ ]:
def compute_self_other_beta_corr(results, region):
    """Pearson correlation between mean self and other beta vectors per neuron."""
    df_self  = results.get(f"{region}_self",  {}).get("coef")
    df_other = results.get(f"{region}_other", {}).get("coef")
    if df_self is None or df_other is None or df_self.empty or df_other.empty:
        return pd.DataFrame()
    common = df_self.index.intersection(df_other.index)
    df_self  = df_self.loc[common].sort_index()
    df_other = df_other.loc[common].sort_index()
    corrs = [pearsonr(df_self.loc[i], df_other.loc[i])[0] for i in df_self.index]
    return pd.DataFrame({"neuron_index": df_self.index, "true_corr": corrs, "region": region.upper()})


def clean_XY(X, Y, label=""):
    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"{label}: X rows ({X.shape[0]}) != Y rows ({Y.shape[0]})")
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(Y).any(axis=1))
    return X[mask], Y[mask], mask


def patient_beta_outputs_complete(patient_ID, regions):
    patient_dir = os.path.join(RESULTS_ROOT, patient_ID)
    if not os.path.exists(os.path.join(patient_dir, "ALL_CONDITIONS_RESULTS.pkl")):
        return False
    for region in regions:
        corr_path = os.path.join(patient_dir, f"{region.upper()}_SELF_OTHER_CORRELATION.csv")
        # Some patients genuinely lack self/other data for a region; the saved pkl is still the main checkpoint.
        if os.path.exists(corr_path):
            continue
    return True


# ---- Main loop ----
pipeline_corr_dfs = []
pipeline_results = {}

for cfg in PATIENTS:
    patient_ID    = cfg["patient_ID"]
    patient       = cfg["patient"]
    region_ranges = cfg["region_ranges"]
    regions       = list(region_ranges.keys())   # patient-specific regions

    print(f"\n{'='*50}")
    print(f"Patient: {patient_ID}  |  regions: {regions}")
    print(f"{'='*50}")

    if RESUME_COMPLETED_PATIENTS and patient_beta_outputs_complete(patient_ID, regions):
        print("  Skipping beta regression — saved outputs already exist")
        continue

    # --- Paths ---
    try:
        spike_base_dir, duration_file = prepare_spike_windows_for_patient(
            patient_ID=patient_ID, patient=patient, region_ranges=region_ranges
        )
    except Exception as e:
        print(f"  Skipping — spike-window prep failed: {e}")
        continue

    llama_npy     = os.path.join(EMBED_CACHE_DIR, f"{patient_ID}_{MODEL_TAG}_word_emb_layers.npy")
    bert_meta_csv = os.path.join(
        BERT_EMBED_DIR, f"{patient_ID}_words_english_only",
        f"{patient_ID}_aligned_embeddings_withNP.csv"
    )

    # --- File checks ---
    missing = [p for p in [spike_base_dir, duration_file, llama_npy, bert_meta_csv]
               if not os.path.exists(p)]
    if missing:
        print(f"  Skipping — missing:\n" + "\n".join(f"    {p}" for p in missing))
        continue
    try:
        # --- Load embeddings → X ---
        X_self, X_other, df_metadata, _, _ = get_self_and_other_features(
            patient_ID, duration_file, n_components=N_COMPONENTS, target_speaker=TARGET_SPEAKER
        )

        # --- Load spikes (only for this patient's regions) ---
        spike_data = load_spike_data(
            speakers=SPEAKERS, regions=regions,
            self_base_dir=spike_base_dir, mode="auto"
        )
        if not spike_data:
            print("  Skipping — no spikes loaded")
            continue

        # --- Build Y ---
        region_Ys = build_Y_matrices(df_metadata, spike_data, regions)

        # --- Align and clean X/Y ---
        X_dict, Y_dict = {}, {}
        for region in regions:
            for cond in ["self", "other"]:
                key = f"{region}_{cond}"
                Y = region_Ys.get(region, {}).get(cond)
                X = X_self if cond == "self" else X_other
                if Y is None or Y.ndim < 2 or Y.shape[0] == 0:
                    continue
                try:
                    Xc, Yc, _ = clean_XY(X, Y, label=key)
                except ValueError as e:
                    print(f"  {key}: {e}")
                    continue
                if Xc.shape[0] == 0:
                    continue
                X_dict[key] = Xc
                Y_dict[key] = Yc
                print(f"  {key}: X={Xc.shape}  Y={Yc.shape}")

        if not X_dict:
            print("  Skipping — no valid conditions")
            continue

        # --- Run regression ---
        results = run_all_conditions_for_patient(
            X_dict=X_dict, Y_dict=Y_dict,
            patient_id=patient_ID, region_list=regions,
            n_semantic_dims=N_COMPONENTS, n_iterations=N_ITERATIONS,
            n_shuf_x=N_X_SHUFFLE_NULLS
        )
        pipeline_results[patient_ID] = results

        # --- Beta correlations ---
        patient_dir = os.path.join(RESULTS_ROOT, patient_ID)
        for region in regions:
            try:
                corr_df = compute_self_other_beta_corr(results, region)
                if corr_df.empty:
                    continue
                corr_df["patient"] = patient_ID
                pipeline_corr_dfs.append(corr_df.copy())
                corr_df.to_csv(
                    os.path.join(patient_dir, f"{region.upper()}_SELF_OTHER_CORRELATION.csv"),
                    index=False
                )
                print(f"  Beta corr saved: {region} ({len(corr_df)} neurons)")
            except Exception as e:
                print(f"  Beta corr failed {region}: {e}")

        # --- Reliability ---
        reliability_mod = load_reliability_module()
        run_beta_reliability_all_neurons = reliability_mod.run_beta_reliability_all_neurons
        ReliabilityConfig = reliability_mod.ReliabilityConfig
        reliability_cfg = ReliabilityConfig(
            n_null=N_NULLS_RELIABILITY,
            alphas=tuple(float(a) for a in ALPHAS),
            n_jobs=N_JOBS_RELIABILITY,
        )
        for region in regions:
            ks, ko = f"{region}_self", f"{region}_other"
            if ks not in X_dict or ko not in X_dict:
                continue
            print(f"  Running reliability: {region}")
            try:
                rel = run_beta_reliability_all_neurons(
                    X_self=X_dict[ks], X_other=X_dict[ko],
                    Y_self=Y_dict[ks], Y_other=Y_dict[ko],
                    cfg=reliability_cfg,
                    verbose=True,
                )
                rel_path = os.path.join(patient_dir, f"{region.upper()}_RELIABILITY_RESULTS.pkl")
                with open(rel_path, "wb") as f:
                    pickle.dump(rel, f)
                print(f"  Reliability saved: {region}")
            except Exception as e:
                print(f"  Reliability failed {region}: {e}")

    except Exception as e:
        print(f"  ERROR: {e}")
        continue


In [ ]:
import glob, os
sorted(glob.glob(os.path.join(RESULTS_ROOT, "*", "ALL_CONDITIONS_RESULTS.pkl")))

## 6 — Aggregate Beta Correlations

In [ ]:
all_corr_dfs = []

for cfg in PATIENTS:
    patient_ID = cfg["patient_ID"]
    for region in REGIONS:
        fpath = os.path.join(RESULTS_ROOT, patient_ID, f"{region.upper()}_SELF_OTHER_CORRELATION.csv")
        if not os.path.exists(fpath):
            continue
        df = pd.read_csv(fpath)
        if "patient" not in df.columns:
            df["patient"] = patient_ID
        all_corr_dfs.append(df)

if not all_corr_dfs and "pipeline_corr_dfs" in globals() and pipeline_corr_dfs:
    all_corr_dfs = pipeline_corr_dfs

combined_df = pd.concat(all_corr_dfs, ignore_index=True) if all_corr_dfs else pd.DataFrame()
if combined_df.empty or "region" not in combined_df.columns or "true_corr" not in combined_df.columns:
    print("No beta-correlation results found yet. Run the main pipeline cell until it prints 'Beta corr saved', then rerun this summary cell.")
    print(f"Checked RESULTS_ROOT: {RESULTS_ROOT}")
else:
    print(combined_df.groupby("region")["true_corr"].describe().round(3))


In [ ]:
if combined_df.empty or "region" not in combined_df.columns or "true_corr" not in combined_df.columns:
    print("No beta-correlation data to plot yet.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, (region, color) in zip(axes, [("HIPPOCAMPUS", "coral"), ("ACC", "mediumseagreen")]):
        df_r = combined_df[combined_df["region"].str.upper() == region].copy()
        df_r = df_r.sort_values("true_corr", ascending=False).reset_index(drop=True)
        df_r["rank"] = df_r.index + 1

        sns.barplot(data=df_r, x="rank", y="true_corr", color=color, ax=ax)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"Self-Other Beta Correlation — {region}\nLlama 3.1-8B Layer {LLAMA_LAYER}")
        ax.set_xlabel("Neuron rank")
        ax.set_ylabel("Pearson r (self beta vs other beta)")
        ax.tick_params(axis="x", labelbottom=False)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_ROOT, f"beta_corr_llama31_L{LLAMA_LAYER}.png"), dpi=150, bbox_inches="tight")
    plt.show()


## 7 — Reliability-only recovery

Use this if the main pipeline already saved beta correlations/results but reliability failed.


In [ ]:
reliability_mod = load_reliability_module(force_reload=True)
run_beta_reliability_all_neurons = reliability_mod.run_beta_reliability_all_neurons
ReliabilityConfig = reliability_mod.ReliabilityConfig
reliability_cfg = ReliabilityConfig(
    n_null=N_NULLS_RELIABILITY,
    alphas=tuple(float(a) for a in ALPHAS),
    n_jobs=N_JOBS_RELIABILITY,
)

for cfg in PATIENTS:
    patient_ID    = cfg["patient_ID"]
    patient       = cfg["patient"]
    region_ranges = cfg["region_ranges"]
    regions       = list(region_ranges.keys())
    patient_dir   = os.path.join(RESULTS_ROOT, patient_ID)

    if not os.path.exists(os.path.join(patient_dir, "ALL_CONDITIONS_RESULTS.pkl")):
        print(f"{patient_ID}: no saved regression results; skipping")
        continue

    print(f"\nReliability recovery: {patient_ID}")
    try:
        spike_base_dir, duration_file = prepare_spike_windows_for_patient(
            patient_ID=patient_ID, patient=patient, region_ranges=region_ranges
        )
        X_self, X_other, df_metadata, _, _ = get_self_and_other_features(
            patient_ID, duration_file, n_components=N_COMPONENTS, target_speaker=TARGET_SPEAKER
        )
        spike_data = load_spike_data(
            speakers=SPEAKERS, regions=regions,
            self_base_dir=spike_base_dir, mode="auto"
        )
        region_Ys = build_Y_matrices(df_metadata, spike_data, regions)

        X_dict, Y_dict = {}, {}
        for region in regions:
            for cond in ["self", "other"]:
                key = f"{region}_{cond}"
                Y = region_Ys.get(region, {}).get(cond)
                X = X_self if cond == "self" else X_other
                if Y is None or Y.ndim < 2 or Y.shape[0] == 0:
                    continue
                try:
                    Xc, Yc, _ = clean_XY(X, Y, label=key)
                except ValueError as e:
                    print(f"  {key}: {e}")
                    continue
                if Xc.shape[0] > 0:
                    X_dict[key] = Xc
                    Y_dict[key] = Yc

        for region in regions:
            ks, ko = f"{region}_self", f"{region}_other"
            if ks not in X_dict or ko not in X_dict:
                print(f"  {region}: missing self/other data; skipping reliability")
                continue
            rel_path = os.path.join(patient_dir, f"{region.upper()}_RELIABILITY_RESULTS.pkl")
            print(f"  Running reliability: {region}")
            rel = run_beta_reliability_all_neurons(
                X_self=X_dict[ks], X_other=X_dict[ko],
                Y_self=Y_dict[ks], Y_other=Y_dict[ko],
                cfg=reliability_cfg,
                verbose=True,
            )
            with open(rel_path, "wb") as f:
                pickle.dump(rel, f)
            print(f"  Reliability saved: {rel_path}")
    except Exception as e:
        print(f"  Reliability recovery failed {patient_ID}: {e}")


## 7 — Reliability Analysis

In [ ]:
import sys
import importlib.util

PROJECT_ROOT = "/scratch/aniluchavez/hippocampal-speaker-semantics"
RELIABILITY_MODULE_PATH = os.path.join(PROJECT_ROOT, "neural_encoding", "reliability.py")

if "nn_reliability" in sys.modules:
    reliability_mod = sys.modules["nn_reliability"]
else:
    spec = importlib.util.spec_from_file_location("nn_reliability", RELIABILITY_MODULE_PATH)
    reliability_mod = importlib.util.module_from_spec(spec)
    sys.modules["nn_reliability"] = reliability_mod
    spec.loader.exec_module(reliability_mod)

aggregate_null_distribution_from_neurons = reliability_mod.aggregate_null_distribution_from_neurons
analyze_reliability_results = reliability_mod.analyze_reliability_results

all_rel_by_region = {region: [] for region in REGIONS}

for cfg in PATIENTS:
    patient_ID = cfg["patient_ID"]
    for region in REGIONS:
        fpath = os.path.join(RESULTS_ROOT, patient_ID, f"{region.upper()}_RELIABILITY_RESULTS.pkl")
        if not os.path.exists(fpath):
            continue
        with open(fpath, "rb") as f:
            rel = pickle.load(f)
        all_rel_by_region[region].extend(rel)

for region, rel_list in all_rel_by_region.items():
    if not rel_list:
        print(f"{region}: no reliability results")
        continue
    true_r_cross = [r["r_cross"] for r in rel_list if not np.isnan(r.get("r_cross", np.nan))]
    null_pool    = [v for r in rel_list for v in r.get("null_distribution", []) if not np.isnan(v)]

    pct = float(np.mean(np.array(true_r_cross) > np.percentile(null_pool, 95))) * 100 if null_pool else np.nan
    print(f"{region}: n={len(true_r_cross)}  mean r_cross={np.mean(true_r_cross):.3f}  "
          f"above 95th null={pct:.1f}%")


In [ ]:
# Three reliability violins: speaking, listening, and cross-condition beta correlation.
violin_rows = []

for cfg in PATIENTS:
    patient_ID = cfg["patient_ID"]
    for region in REGIONS:
        fpath = os.path.join(RESULTS_ROOT, patient_ID, f"{region.upper()}_RELIABILITY_RESULTS.pkl")
        if not os.path.exists(fpath):
            continue
        with open(fpath, "rb") as f:
            rel = pickle.load(f)
        for r in rel:
            neuron = r.get("neuron", np.nan)
            values = {
                "Speaking": r.get("self_reliability_mean", np.nan),
                "Listening": r.get("other_reliability_mean", np.nan),
                "Cross correlation": r.get("r_cross", np.nan),
            }
            for metric, value in values.items():
                violin_rows.append({
                    "patient": patient_ID,
                    "region": region.upper(),
                    "neuron": neuron,
                    "metric": metric,
                    "value": value,
                })

violin_df = pd.DataFrame(violin_rows)
if violin_df.empty:
    print("No reliability result files found yet. Run the Reliability-only recovery cell first.")
    print(f"Checked RESULTS_ROOT: {RESULTS_ROOT}")
elif {"Speaking", "Listening"}.isdisjoint(set(violin_df["metric"])) or violin_df["value"].notna().sum() == 0:
    print("Reliability files exist, but they do not include speaking/listening reliability yet.")
    print("Rerun the first import cell, then rerun the Reliability-only recovery cell to regenerate them.")
else:
    order = ["Speaking", "Listening", "Cross correlation"]
    violin_df["metric"] = pd.Categorical(violin_df["metric"], categories=order, ordered=True)
    summary = (
        violin_df.dropna(subset=["value"])
        .groupby(["region", "metric"], observed=True)["value"]
        .agg(["count", "mean", "median", "std"])
        .round(3)
    )
    print(summary)

    regions_present = sorted(violin_df["region"].dropna().unique())
    fig, axes = plt.subplots(1, len(regions_present), figsize=(6.5 * len(regions_present), 5.5), sharey=True)
    if len(regions_present) == 1:
        axes = [axes]

    palette = {
        "Speaking": "#4C78A8",
        "Listening": "#F58518",
        "Cross correlation": "#54A24B",
    }
    for ax, region in zip(axes, regions_present):
        rdf = violin_df[violin_df["region"] == region].dropna(subset=["value"])
        sns.violinplot(
            data=rdf, x="metric", y="value", order=order,
            palette=palette, inner="quartile", cut=0, linewidth=1, ax=ax
        )
        sns.stripplot(
            data=rdf, x="metric", y="value", order=order,
            color="black", alpha=0.22, size=2.5, jitter=0.18, ax=ax
        )
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"{region} reliability")
        ax.set_xlabel("")
        ax.set_ylabel("Reliability / correlation")
        ax.tick_params(axis="x", rotation=20)

    fig.suptitle(f"Speaking, listening, and cross beta correlation — Llama 3.1-8B Layer {LLAMA_LAYER}", y=1.03)
    plt.tight_layout()
    plot_path = os.path.join(RESULTS_ROOT, f"three_violin_reliability_llama31_L{LLAMA_LAYER}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    print(f"Saved reliability violin plot: {plot_path}")
    plt.show()


In [ ]:
import glob, os
sorted(glob.glob(os.path.join(RESULTS_ROOT, "*", "*SELF_OTHER_CORRELATION.csv")))

In [ ]:
sorted(glob.glob(os.path.join(RESULTS_ROOT, "*", "*RELIABILITY_RESULTS.pkl")))